## Cell 1 — Import and load the embedding model

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Model loaded successfully")

D:\Gen AI\Project 3\ecommerce-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3035.23it/s]


Model loaded successfully


## Cell 2 — Create three sentences

In [2]:
sentences = [
    "What is your return policy?",
    "Can I return a product after buying it?",
    "Show me Nike shoes under 5000 rupees"
]

sentences

['What is your return policy?',
 'Can I return a product after buying it?',
 'Show me Nike shoes under 5000 rupees']

## Cell 3 — Convert the sentences into embeddings

In [3]:
embeddings = model.encode(sentences)

print(embeddings)

[[-0.00747141  0.04949777  0.04888179 ... -0.0640774  -0.06426486
  -0.02677878]
 [-0.048869   -0.00987116  0.0066841  ... -0.07794641  0.07136469
  -0.018107  ]
 [-0.04164219  0.02853357  0.00936462 ... -0.12277708 -0.06219479
  -0.01058846]]


## Cell 4 — Inspect the shape

In [4]:
print("Embedding shape:", embeddings.shape)

print("\nNumber of sentences:", embeddings.shape[0])
print("Dimensions per sentence:", embeddings.shape[1])

Embedding shape: (3, 384)

Number of sentences: 3
Dimensions per sentence: 384


## Cell 5 — Calculate cosine similarity manually

In [5]:
import numpy as np

def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )

sim_1_2 = cosine_similarity(embeddings[0], embeddings[1])
sim_1_3 = cosine_similarity(embeddings[0], embeddings[2])

print("Sentence 1 vs Sentence 2:", sim_1_2)
print("Sentence 1 vs Sentence 3:", sim_1_3)

Sentence 1 vs Sentence 2: 0.5014082
Sentence 1 vs Sentence 3: 0.12204574


## Cell 6 — Use SentenceTransformers' built-in utility

In [6]:
from sentence_transformers import util

score_1_2 = util.cos_sim(embeddings[0], embeddings[1])
score_1_3 = util.cos_sim(embeddings[0], embeddings[2])

print("Return policy vs Return product:", score_1_2.item())
print("Return policy vs Nike shoes:", score_1_3.item())

Return policy vs Return product: 0.5014082193374634
Return policy vs Nike shoes: 0.12204574793577194


## Cell 7 — Compare everything at once

In [7]:
similarity_matrix = util.cos_sim(embeddings, embeddings)

print(similarity_matrix)

tensor([[1.0000, 0.5014, 0.1220],
        [0.5014, 1.0000, 0.1822],
        [0.1220, 0.1822, 1.0000]])


In [8]:
sentences = [
    "What is your return policy?",
    "Can I send back an item I purchased?",
    "Return to the home page"
]

embeddings_test = model.encode(sentences)

similarities = util.cos_sim(
    embeddings_test[0],
    embeddings_test[1:]
)

print("Return policy vs send back item:",
      similarities[0][0].item())

print("Return policy vs return home:",
      similarities[0][1].item())

Return policy vs send back item: 0.39958298206329346
Return policy vs return home: 0.3127632141113281


# Keyword search compares words. Embeddings compare semantic meaning.

## Step 3 — Why semantic search beats keyword search

In [9]:
test_sentences = [
    "What is your return policy?",
    "Can I send back an item I purchased?",
    "Return to home page",
    "Show me Puma shoes under 4000"
]

test_embeddings = model.encode(test_sentences)

scores = util.cos_sim(test_embeddings, test_embeddings)

print(scores)

tensor([[ 1.0000,  0.3996,  0.3238,  0.0978],
        [ 0.3996,  1.0000,  0.2293,  0.0427],
        [ 0.3238,  0.2293,  1.0000, -0.0445],
        [ 0.0978,  0.0427, -0.0445,  1.0000]])


## Step 4 — ChromaDB

### Cell 9 — Import ChromaDB

In [11]:
import chromadb
from chromadb.utils import embedding_functions

In [12]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1388.57it/s]


## Step 5 — Persistent ChromaDB

In [13]:
chroma_client = chromadb.PersistentClient(
    path="../data/chroma_db"
)

In [14]:
chroma_client = chromadb.PersistentClient(
    path="../data/chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="faqs",
    embedding_function=ef
)

print(collection)

Collection(name=faqs)


## chromadb.Client() → mostly session/in-memory style usage

## PersistentClient() → embeddings saved on disk

## Step 6 — Manually ingest 3 FAQs first

In [15]:
documents = [
    "What is the return policy of the products?",
    "How can I track my order?",
    "What payment methods are accepted?"
]

metadatas = [
    {"answer": "You can return products within 30 days of delivery."},
    {"answer": "You can track your order using the tracking link sent to your email."},
    {"answer": "We accept credit/debit cards, net banking, UPI, and cash on delivery."}
]

ids = [
    "faq_1",
    "faq_2",
    "faq_3"
]

In [16]:
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("FAQ count:", collection.count())

FAQ count: 3


document
"What payment methods are accepted?"

        ↓ embedding

[384 numbers]

        ↓

stored inside ChromaDB

ID:
faq_3

Metadata:
{
    "answer":
    "We accept credit/debit cards..."
}

## Step 7 — Semantic search

In [17]:
query = "Can I pay using UPI?"

results = collection.query(
    query_texts=[query],
    n_results=2
)

results

{'ids': [['faq_3', 'faq_1']],
 'embeddings': None,
 'documents': [['What payment methods are accepted?',
   'What is the return policy of the products?']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'answer': 'We accept credit/debit cards, net banking, UPI, and cash on delivery.'},
   {'answer': 'You can return products within 30 days of delivery.'}]],
 'distances': [[0.5809004902839661, 0.7979933619499207]]}

## smaller distance
    ↓
## more similar

## larger distance
    ↓
## less similar

## Step 8 — Load the actual FAQ dataset

### Cell 13 — Read and inspect the FAQ data

In [18]:
import pandas as pd

faq_path = "../data/raw/faq_data.csv"

df = pd.read_csv(faq_path)

print("Shape:", df.shape)
display(df.head())

Shape: (10, 2)


,question,answer
0,What is the return policy of the products?,You can return products within 30 days of deli...
1,Do I get discount with the HDFC credit card?,"Yes, HDFC credit card users get a 10% discount..."
2,How can I track my order?,You can track your order using the tracking li...
3,What payment methods are accepted?,"We accept credit/debit cards, net banking, UPI..."
4,How long does it take to process a refund?,Refunds are processed within 5-7 business days...


## Step 9 — Prepare data for Chroma

In [19]:
documents = df["question"].tolist()

metadatas = [
    {"answer": answer}
    for answer in df["answer"].tolist()
]

ids = [
    f"faq_{i}"
    for i in range(len(df))
]

print("Documents:", len(documents))
print("Metadata:", len(metadatas))
print("IDs:", len(ids))

print("\nExample document:")
print(documents[0])

print("\nExample metadata:")
print(metadatas[0])

print("\nExample ID:")
print(ids[0])

Documents: 10
Metadata: 10
IDs: 10

Example document:
What is the return policy of the products?

Example metadata:
{'answer': 'You can return products within 30 days of delivery.'}

Example ID:
faq_0


In [20]:
## Step 10 — Replace demo collection with real data

In [21]:
chroma_client.delete_collection(name="faqs")

collection = chroma_client.create_collection(
    name="faqs",
    embedding_function=ef
)

print("Fresh collection created.")

Fresh collection created.


In [22]:
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("FAQs ingested:", collection.count())

FAQs ingested: 10


In [23]:
## Step 11 — Test semantic retrieval properly

In [24]:
query = "My product arrived broken. What should I do?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i in range(len(results["documents"][0])):
    print(f"Result {i + 1}")
    print("Question :", results["documents"][0][i])
    print("Answer   :", results["metadatas"][0][i]["answer"])
    print("Distance :", results["distances"][0][i])
    print("-" * 60)

Result 1
Question : What should I do if I receive a damaged product?
Answer   : Contact our support team within 48 hours for a replacement or refund.
Distance : 0.17408961057662964
------------------------------------------------------------
Result 2
Question : How can I track my order?
Answer   : You can track your order using the tracking link sent to your email.
Distance : 0.686797022819519
------------------------------------------------------------
Result 3
Question : What is the return policy of the products?
Answer   : You can return products within 30 days of delivery.
Distance : 0.7102675437927246
------------------------------------------------------------


# Step 12 — Connect Groq

In [25]:
import os
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

env_path = Path("..") / ".env"
load_dotenv(env_path)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL")

client = Groq(api_key=GROQ_API_KEY)

print("Model:", GROQ_MODEL)
print("API key loaded:", bool(GROQ_API_KEY))

Model: openai/gpt-oss-20b
API key loaded: True


In [26]:
## Step 13 — Turn Chroma retrieval into a function

In [27]:
def get_relevant_faqs(query, top_k=3):
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    
    return results

In [28]:
query = "Can I return something that arrived damaged?"

results = get_relevant_faqs(query)

results

{'ids': [['faq_8', 'faq_0', 'faq_4']],
 'embeddings': None,
 'documents': [['What should I do if I receive a damaged product?',
   'What is the return policy of the products?',
   'How long does it take to process a refund?']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'answer': 'Contact our support team within 48 hours for a replacement or refund.'},
   {'answer': 'You can return products within 30 days of delivery.'},
   {'answer': 'Refunds are processed within 5-7 business days after the return is received.'}]],
 'distances': [[0.279413104057312, 0.6320309042930603, 0.6815190315246582]]}

In [29]:
## Step 14 — Build the context

In [30]:
def build_context(results):
    context_parts = []
    
    for question, metadata in zip(
        results["documents"][0],
        results["metadatas"][0]
    ):
        answer = metadata["answer"]
        
        context_parts.append(
            f"FAQ Question: {question}\n"
            f"FAQ Answer: {answer}"
        )
    
    return "\n\n".join(context_parts)

In [31]:
context = build_context(results)

print(context)


FAQ Question: What should I do if I receive a damaged product?
FAQ Answer: Contact our support team within 48 hours for a replacement or refund.

FAQ Question: What is the return policy of the products?
FAQ Answer: You can return products within 30 days of delivery.

FAQ Question: How long does it take to process a refund?
FAQ Answer: Refunds are processed within 5-7 business days after the return is received.


## Step 15 — Send question + context to Groq

In [32]:
def generate_faq_answer(query, context):
    prompt = f"""
You are a helpful e-commerce customer support assistant.

Answer the user's question using ONLY the FAQ context provided below.

If the answer cannot be found in the context, say:
"I don't know based on the available FAQ information."

Do not invent policies, prices, timelines, or other information.

FAQ CONTEXT:
{context}

USER QUESTION:
{query}

ANSWER:
"""
    
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )
    
    return response.choices[0].message.content.strip()

In [33]:
answer = generate_faq_answer(query, context)

print(answer)

Yes – if a product arrives damaged you should contact our support team within 48 hours. They can arrange a replacement or refund for you.


## Step 16 — Create the complete FAQ chain

In [34]:
def faq_chain(query, top_k=3):
    results = get_relevant_faqs(
        query=query,
        top_k=top_k
    )
    
    context = build_context(results)
    
    answer = generate_faq_answer(
        query=query,
        context=context
    )
    
    return answer

In [35]:
query = "Do you accept cash when the product is delivered?"

answer = faq_chain(query)

print(answer)

Yes, we accept cash on delivery.


In [36]:
## Step 17 — Inspect the entire RAG pipelinem

In [37]:
query = "How long will it take to get my money back?"

results = get_relevant_faqs(query)

print("USER QUERY:")
print(query)

print("\n--- RETRIEVED FAQs ---")

for i, (question, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\nResult {i}")
    print("Question :", question)
    print("Answer   :", metadata["answer"])
    print("Distance :", round(distance, 4))

context = build_context(results)

print("\n--- LLM ANSWER ---")
print(generate_faq_answer(query, context))

USER QUERY:
How long will it take to get my money back?

--- RETRIEVED FAQs ---

Result 1
Question : How long does it take to process a refund?
Answer   : Refunds are processed within 5-7 business days after the return is received.
Distance : 0.2504

Result 2
Question : What is the return policy of the products?
Answer   : You can return products within 30 days of delivery.
Distance : 0.7004

Result 3
Question : What payment methods are accepted?
Answer   : We accept credit/debit cards, net banking, UPI, and cash on delivery.
Distance : 0.7426

--- LLM ANSWER ---
Refunds are processed within 5-7 business days after the return is received.


In [ ]:
aa